# Init

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from pyspark.sql import SparkSession

%pip install openpyxl

print("A iniciar o pipeline de extração...")

def sanitize_column(col):
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = col.encode('ascii', 'ignore').decode()
    col = col.lower()
    return col

URL_ANP = "https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/levantamento-de-precos-de-combustiveis-ultimas-semanas-pesquisadas"
caminho_volume = "/Volumes/projeto_combustiveis/bronze/raw/resumo_semanal_recente.xlsx"

# Web Scraping

In [0]:
print("A aceder à página da ANP e a procurar o relatório exato...")
resposta = requests.get(URL_ANP)
soup = BeautifulSoup(resposta.content, 'html.parser')

link_excel = None
texto_alvo = "preços médios semanais: brasil, regiões, estados e municípios"

for tag_a in soup.find_all('a', href=True):
    href = tag_a['href']
    texto_limpo = " ".join(tag_a.text.split()).lower()
    
    if '.xlsx' in href.lower() and texto_alvo in texto_limpo:
        link_excel = href
        break

if not link_excel:
    raise Exception(f"Não foi possível encontrar o ficheiro com o nome '{texto_alvo}' na página.")

if not link_excel.startswith('http'):
    link_excel = "https://www.gov.br" + link_excel

print(f"Link da semana atual encontrado: {link_excel}")

# Download da Planilha

In [0]:
print("A fazer o download do ficheiro...")
arquivo_excel = requests.get(link_excel)

with open(caminho_volume, "wb") as f:
    f.write(arquivo_excel.content)
    
print(f"Ficheiro guardado com sucesso no volume: {caminho_volume}")

# Ingestão Incremental na Camada Bronze

In [0]:
xls = pd.ExcelFile(caminho_volume)

mapeamento_tabelas = {
    "BRASIL": "semanal_brasil",
    "REGIOES": "semanal_regioes",
    "ESTADOS": "semanal_estados",
    "MUNICIPIOS": "semanal_municipios",
    "CAPITAIS": "semanal_capitais"
}

for aba_excel, tabela_bronze in mapeamento_tabelas.items():
    aba_existente = None
    for sheet in xls.sheet_names:
        if sanitize_column(sheet) == sanitize_column(aba_excel):
            aba_existente = sheet
            break

    if aba_existente:
        print(f"\nA processar a aba: {aba_existente}...")
        
        df_pandas = pd.read_excel(xls, sheet_name=aba_existente, skiprows=9)
        df_pandas = df_pandas.astype(str)
        df_pandas.columns = [sanitize_column(col) for col in df_pandas.columns]
        
        data_nova = df_pandas['data_inicial'].iloc[0]
        nome_tabela_destino = f"projeto_combustiveis.bronze.{tabela_bronze}"
        
        try:
            df_verificacao = spark.sql(f"SELECT 1 FROM {nome_tabela_destino} WHERE data_inicial = '{data_nova}' LIMIT 1")
            ja_existe = df_verificacao.count() > 0
        except:
            ja_existe = False
            
        if ja_existe:
             print(f"AVISO: Dados da semana de '{data_nova}' já existem na tabela '{tabela_bronze}'. O append foi ignorado.")
        else:
            df_spark = spark.createDataFrame(df_pandas)
            df_spark.write.mode("append").saveAsTable(nome_tabela_destino)
            print(f" -> OK: Novos dados ({data_nova}) adicionados com sucesso à tabela {tabela_bronze}!")

        print(f" -> Dados adicionados com sucesso na tabela: {tabela_bronze}")